# AWREAS Scoring Evaluator on ArgRewrite V.2
**Metrics:** Quadratic Weighted Kappa (QWK) · Mean Absolute Error (MAE) · Spearman Correlation

This notebook benchmarks AWREAS scoring performance on the **ArgRewrite V.2** corpus directly inside Colab.
It loads the Draft 2 and Draft 3 texts, matches them with their corresponding expert gold scores (out of 40) from the dataset metadata, runs the supervisor evaluation pipeline, and computes agreement metrics.

**Pipeline per essay draft:**
1. Load Draft 2 and Draft 3 pairs from the cloned ArgRewrite repository.
2. Load human expert scores from `scores.xlsx` for each draft.
3. Run the supervisor evaluation pipeline on both drafts (60 total evaluations).
4. Rescale the predicted scores (0–100 scale) to the human range (10–40) using standard and calibrated rescaling.
5. Compute QWK, MAE, and Spearman Correlation with human gold scores.

**Before running:** Upload `academic-review-system.zip` to Google Drive.

## Cell 1 — Mount Drive and Extract Project

In [ ]:
from google.colab import drive
import os, sys, glob

drive.mount('/content/drive')

zip_path    = '/content/drive/MyDrive/academic-review-system.zip'
extract_dir = '/content/academic-review-system'

if os.path.exists(zip_path):
    print('Found zip — extracting...')
    !unzip -q -o "{zip_path}" -d "{extract_dir}"
    print('Extraction complete.')
else:
    extract_dir = '/content/drive/MyDrive/academic-review-system'
    if not os.path.exists(extract_dir):
        raise FileNotFoundError('Upload academic-review-system.zip to Google Drive first.')

toml_paths = glob.glob(os.path.join(extract_dir, '**/pyproject.toml'), recursive=True)
if not toml_paths:
    raise FileNotFoundError('Cannot locate pyproject.toml inside extracted files.')

project_root = os.path.dirname(toml_paths[0])
print(f'Project root: {project_root}')
sys.path.insert(0, project_root)
%cd "{project_root}"

## Cell 2 — Install Dependencies

In [ ]:
!pip install -q .
!pip install -q google-genai openpyxl pandas
print('Dependencies installed.')

## Cell 3 — Configure LLM Provider and API Key

In [ ]:
import getpass

print('Available providers: gemini | openrouter | openai | qwen')
provider = input('Enter LLM provider: ').strip().lower()
if provider not in ('gemini', 'openrouter', 'openai', 'qwen'):
    print("Unrecognised — defaulting to 'openai'.")
    provider = 'openai'

api_key = getpass.getpass(f'Paste your {provider.upper()} API key: ').strip()
if not api_key:
    raise ValueError('API key is required.')

MODEL_DEFAULTS = {
    'openai':     'gpt-4o-mini',
    'openrouter': 'anthropic/claude-sonnet-4.6',
    'gemini':     'gemini-2.5-flash',
    'qwen':       'qwen3.7-max',
}
default_model = MODEL_DEFAULTS[provider]
model_input   = input(f'Model name [{default_model}]: ').strip()
model_name    = model_input if model_input else default_model

print(f'\nProvider : {provider}')
print(f'Model    : {model_name}')
print('Ready.')

## Cell 4 — Initialise AWREAS

In [ ]:
# Clear any stale cached imports from a previous run in this session
for mod in [k for k in sys.modules if k.startswith('app')]:
    del sys.modules[mod]

from app.core.config import settings
settings.ENABLE_WEB_RESEARCH = False  # prevents slow external lookups during benchmarking
settings.LLM_REQUEST_TIMEOUT = 60.0

settings.WORKER_MODEL_NAME    = model_name
settings.SYNTHESIS_MODEL_NAME = model_name
settings.REVIEW_MODEL_NAME    = model_name
settings.REVISION_MODEL_NAME  = model_name

from app.llm.client import LLMClient
from app.supervisor.agent import SupervisorAgent

llm_client = LLMClient(api_key=api_key, provider=provider, default_model=model_name)
supervisor  = SupervisorAgent(llm_client=llm_client)

print(f'SupervisorAgent ready (provider={provider}, model={model_name}).')
print(f'LLM available: {llm_client.available}')

## Cell 5 — Clone and Parse ArgRewrite V.2 Corpus & Load Gold Scores

In [ ]:
# 1. Run the dataset preparation script to clone and parse the corpus
!python scripts/prepare_argrewrite_benchmark.py

# 2. Load the parsed pairs
import json
import pandas as pd
from pathlib import Path

CACHE_PATH = Path('data/argrewrite/pairs.json')
pairs = json.loads(CACHE_PATH.read_text(encoding='utf-8'))

# 3. Load expert gold scores
if os.path.exists('/content'):
    scores_path = Path('/content/ArgRewrite/dataset/meta-data/Scores/scores.xlsx')
else:
    scores_path = Path('scratch/ArgRewrite_clone/dataset/meta-data/Scores/scores.xlsx')

df_scores = pd.read_excel(scores_path)
# Build lookup: {(writer_id, draft_num): avg_total_score}
gold_scores = {}
for _, row in df_scores.iterrows():
    writer = int(row['writer'])
    draft  = int(row['draft'])
    score  = float(row['avg_total'])
    gold_scores[(writer, draft)] = score

print(f'\nLoaded {len(pairs)} essay pairs with Draft 2 + Draft 3.')
print(f'Loaded {len(gold_scores)} human scores from scores.xlsx')

## Cell 6 — Sampling and Run Configuration

In [ ]:
import random

# ---------------------------------------------------------------
# CONFIGURATION — adjust as needed
# ---------------------------------------------------------------
SAMPLE_SIZE    = 30    # Number of essay pairs to evaluate. Set to None for all.
RANDOM_SEED    = 42    # Seed for reproducible sampling
CITATION_STYLE = 'apa' 
GOLD_MIN       = 10    # ArgRewrite theoretical min score
GOLD_MAX       = 40    # ArgRewrite theoretical max score
# ---------------------------------------------------------------

random.seed(RANDOM_SEED)

# Keep only pairs that have gold scores for BOTH draft 2 and draft 3
valid_pairs = []
for p in pairs:
    wid = int(p['id'])
    if (wid, 2) in gold_scores and (wid, 3) in gold_scores:
        valid_pairs.append(p)

if SAMPLE_SIZE is not None and len(valid_pairs) > SAMPLE_SIZE:
    essays_to_run = random.sample(valid_pairs, SAMPLE_SIZE)
else:
    essays_to_run = valid_pairs

print(f'Essays to evaluate : {len(essays_to_run)} pairs (60 evaluations total)')
print(f'Citation style     : {CITATION_STYLE}')

## Cell 7 — Run Scoring Evaluation

In [ ]:
import time, statistics as st

evaluation_records = []  # [{id, draft_num, raw_pred, gold_score, worker_scores, latency, approved}, ...]
failed = []

total_essays = len(essays_to_run)
print(f'Starting evaluation: {total_essays} essay pairs (60 evaluations total)...\n')

for i, pair in enumerate(essays_to_run):
    essay_id = int(pair['id'])
    
    for draft_num, text in [(2, pair['draft2']), (3, pair['draft3'])]:
        try:
            t0 = time.perf_counter()
            context = await supervisor.run_evaluation(content=text, citation_style=CITATION_STYLE)
            latency = (time.perf_counter() - t0) * 1000

            gold_val = gold_scores[(essay_id, draft_num)]

            evaluation_records.append({
                'id': essay_id,
                'draft_num': draft_num,
                'raw_pred': context.overall_score,
                'gold_score': gold_val,
                'worker_scores': dict(context.final_scores),
                'latency': latency,
                'approved': context.critic_review.approved if context.critic_review else True,
            })
            
            print(f'[{i+1}/{total_essays}] ID={essay_id:>3} Draft {draft_num} | Gold={gold_val:>4.1f} | Pred={context.overall_score:>5.2f} | {latency/1000.0:.1f}s')
            
        except Exception as exc:
            failed.append((essay_id, draft_num, str(exc)))
            print(f'  [FAIL] ID={essay_id} Draft {draft_num}: {exc}')

print(f'\nDone. {len(evaluation_records)} drafts evaluated, {len(failed)} failures.')

## Cell 8 — Rescale Predictions to Gold Range (10–40)

In [ ]:
def rescale_standard(score_100, lo=GOLD_MIN, hi=GOLD_MAX):
    return lo + (score_100 / 100.0) * (hi - lo)

raw_preds   = [r['raw_pred']   for r in evaluation_records]
gold_scores_list = [r['gold_score'] for r in evaluation_records]
wall_times  = [r['latency']    for r in evaluation_records]

obs_min   = min(raw_preds) if raw_preds else 0.0
obs_max   = max(raw_preds) if raw_preds else 100.0
obs_range = obs_max - obs_min if obs_max > obs_min else 1.0

for r in evaluation_records:
    r['pred_std'] = rescale_standard(r['raw_pred'])
    r['pred_cal'] = GOLD_MIN + ((r['raw_pred'] - obs_min) / obs_range) * (GOLD_MAX - GOLD_MIN)

print(f'Evaluated Drafts   : {len(evaluation_records)}')
print(f'Observed raw range : [{obs_min:.2f}, {obs_max:.2f}]')
print(f'Gold range         : [{min(gold_scores_list):.1f}, {max(gold_scores_list):.1f}]')

## Cell 9 — Compute QWK, MAE, and Spearman Correlation

In [ ]:
import math

# ── QWK ────────────────────────────────────────────────────────
def quadratic_weighted_kappa(y_true, y_pred, min_r=GOLD_MIN, max_r=GOLD_MAX):
    # Round to integers for QWK
    y_true_int = [int(round(y)) for y in y_true]
    y_pred_int = [int(round(y)) for y in y_pred]
    
    num_ratings = max_r - min_r + 1
    o = [[0] * num_ratings for _ in range(num_ratings)]
    for t, p in zip(y_true_int, y_pred_int):
        t_clamped = max(min_r, min(max_r, t))
        p_clamped = max(min_r, min(max_r, p))
        o[t_clamped - min_r][p_clamped - min_r] += 1
        
    w = [[float((i - j) ** 2) / float((num_ratings - 1) ** 2)
          for j in range(num_ratings)] for i in range(num_ratings)]
    hist_t = [0] * num_ratings
    hist_p = [0] * num_ratings
    for t, p in zip(y_true_int, y_pred_int):
        t_clamped = max(min_r, min(max_r, t))
        p_clamped = max(min_r, min(max_r, p))
        hist_t[t_clamped - min_r] += 1
        hist_p[p_clamped - min_r] += 1
        
    n = sum(hist_t)
    if n == 0:
        return 0.0
    e = [[float(hist_t[i] * hist_p[j]) / n
          for j in range(num_ratings)] for i in range(num_ratings)]
    num = sum(w[i][j] * o[i][j] for i in range(num_ratings) for j in range(num_ratings))
    den = sum(w[i][j] * e[i][j] for i in range(num_ratings) for j in range(num_ratings))
    return 1.0 if den == 0.0 else 1.0 - (num / den)

# ── MAE ────────────────────────────────────────────────────────
def mean_absolute_error(y_true, y_pred):
    return sum(abs(t - p) for t, p in zip(y_true, y_pred)) / len(y_true)

# ── Spearman ───────────────────────────────────────────────────
def spearman_correlation(x, y):
    n = len(x)
    if n < 2:
        return 0.0

    def rank(vals):
        sorted_idx = sorted(range(n), key=lambda i: vals[i])
        r = [0.0] * n
        i = 0
        while i < n:
            j = i
            while j < n - 1 and vals[sorted_idx[j]] == vals[sorted_idx[j + 1]]:
                j += 1
            avg_rank = (i + j) / 2.0 + 1
            for k in range(i, j + 1):
                r[sorted_idx[k]] = avg_rank
            i = j + 1
        return r

    rx, ry   = rank(x), rank(y)
    mean_rx  = sum(rx) / n
    mean_ry  = sum(ry) / n
    num      = sum((rx[i] - mean_rx) * (ry[i] - mean_ry) for i in range(n))
    den_x    = sum((rx[i] - mean_rx) ** 2 for i in range(n))
    den_y    = sum((ry[i] - mean_ry) ** 2 for i in range(n))
    if den_x == 0.0 or den_y == 0.0:
        return 0.0
    return num / math.sqrt(den_x * den_y)

# ── Compute ────────────────────────────────────────────────────
pred_std_list = [r['pred_std'] for r in evaluation_records]
pred_cal_list = [r['pred_cal'] for r in evaluation_records]

qwk_std  = quadratic_weighted_kappa(gold_scores_list, pred_std_list)
mae_std  = mean_absolute_error(gold_scores_list, pred_std_list)
spr_std  = spearman_correlation(gold_scores_list, pred_std_list)

qwk_cal  = quadratic_weighted_kappa(gold_scores_list, pred_cal_list)
mae_cal  = mean_absolute_error(gold_scores_list, pred_cal_list)
spr_cal  = spearman_correlation(gold_scores_list, pred_cal_list)

print('Standard rescaling   [y = 10 + (S/100) × 30]')
print(f'  QWK      : {qwk_std:.4f}')
print(f'  MAE      : {mae_std:.4f}')
print(f'  Spearman : {spr_std:.4f}')
print()
print('Calibrated rescaling [observed range → 10–40]')
print(f'  QWK      : {qwk_cal:.4f}')
print(f'  MAE      : {mae_cal:.4f}')
print(f'  Spearman : {spr_cal:.4f}')

## Cell 10 — Operational Metrics & Worker Breakdown

In [ ]:
def percentile(vals, p):
    xs = sorted(vals)
    if not xs:
        return 0.0
    k  = (len(xs) - 1) * p
    lo = math.floor(k)
    hi = math.ceil(k)
    return xs[lo] if lo == hi else xs[lo] * (hi - k) + xs[hi] * (k - lo)

p50 = percentile(wall_times, 0.50) if wall_times else 0.0
p95 = percentile(wall_times, 0.95) if wall_times else 0.0
mean_lat = st.mean(wall_times) if wall_times else 0.0
approvals = sum(1 for r in evaluation_records if r['approved'])
critic_rate = (approvals / len(evaluation_records)) * 100.0 if evaluation_records else 0.0

print(f'Latency P50       : {p50/1000.0:.2f} s')
print(f'Latency P95       : {p95/1000.0:.2f} s')
print(f'Mean Latency      : {mean_lat/1000.0:.2f} s')
print(f'Critic Approval   : {approvals}/{len(evaluation_records)} ({critic_rate:.1f}%)')
print()

if evaluation_records:
    worker_names = sorted(evaluation_records[0]['worker_scores'].keys())
    print(f'{"Worker Agent":<32} | {"Mean Score":>10} | {"Spearman r":>10}')
    print('-' * 58)
    for w_name in worker_names:
        w_scores = [r['worker_scores'].get(w_name, 0.0) for r in evaluation_records]
        w_corr   = spearman_correlation(gold_scores_list, w_scores)
        print(f'{w_name:<32} | {st.mean(w_scores):>10.2f} | {w_corr:>10.4f}')

## Cell 11 — Full Summary

In [ ]:
print('=' * 62)
print('AWREAS SCORING BENCHMARK — ARG-REWRITE SUMMARY')
print('=' * 62)
print(f'Corpus          : ArgRewrite V.2 (Kashefi et al., 2022)')
print(f'Gold score range: {GOLD_MIN}–{GOLD_MAX}')
print(f'Drafts evaluated: {len(evaluation_records)}')
print(f'Provider / Model: {provider} / {model_name}')
print()
print('Standard Rescaling  [y = 10 + (S/100) × 30]')
print(f'  QWK              : {qwk_std:.4f}')
print(f'  MAE              : {mae_std:.4f}')
print(f'  Spearman rho     : {spr_std:.4f}')
print()
print('Calibrated Rescaling [observed range → 10–40]')
print(f'  QWK              : {qwk_cal:.4f}')
print(f'  MAE              : {mae_cal:.4f}')
print(f'  Spearman rho     : {spr_cal:.4f}')
print(f'  Observed raw rng : [{obs_min:.2f}, {obs_max:.2f}]')
print()
print(f'Critic approval rate        : {critic_rate:.1f}%')
print(f'Wall latency P50 / P95      : {p50/1000.0:.2f} s / {p95/1000.0:.2f} s')
print('=' * 62)

## Cell 12 — Save Results to JSON

In [ ]:
output_file = Path('data/argrewrite/scoring_results_argrewrite.json')
output_file.parent.mkdir(parents=True, exist_ok=True)

summary = {
    'meta': {
        'corpus': 'ArgRewrite V.2',
        'evaluations': len(evaluation_records),
        'provider': provider,
        'model': model_name,
        'gold_range': [GOLD_MIN, GOLD_MAX],
    },
    'aggregate': {
        'qwk_standard': round(qwk_std, 4),
        'mae_standard': round(mae_std, 4),
        'spearman_standard': round(spr_std, 4),
        'qwk_calibrated': round(qwk_cal, 4),
        'mae_calibrated': round(mae_cal, 4),
        'spearman_calibrated': round(spr_cal, 4),
        'critic_approval_pct': round(critic_rate, 2),
        'latency_p50_ms': round(p50, 1),
        'latency_p95_ms': round(p95, 1),
    },
    'results': evaluation_records
}

output_file.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'Saved scoring benchmark to: {output_file}')

## Cell 13 — Back Up Results to Google Drive

In [ ]:
import shutil

drive_dest = '/content/drive/MyDrive/AWREAS_ArgRewrite_Scoring_Results'
os.makedirs(drive_dest, exist_ok=True)
dest = shutil.copy(str(output_file), drive_dest)
print(f'Backed up to Google Drive: {dest}')